# Phase 2: stage the invest pre-registration import

Loads `invest_preregistrations_2022-to-2026-08-25_exported-2026-08-26.csv` (3,193 lines, 3,192
pre-registrants plus one empty trailing line) into `crm_imp_person_accounts` under four batch ids. The account
loaders ignore the suffix, the consent loader reads OptIn/OptOut from it, and no row
ever has to move batches mid-run:

| batch id | _operation | population |
|---|---|---|
| `2026-08-28_prereg_update_optin`  | update | email/extid match, `consent_invest=1` |
| `2026-08-28_prereg_update_optout` | update | email/extid match, `consent_invest=0` |
| `2026-08-28_prereg_insert_optin`  | insert | no match, `consent_invest=1` |
| `2026-08-28_prereg_insert_optout` | insert | no match, `consent_invest=0` |

This notebook only writes to the local MySQL staging table. It never touches Salesforce.

Decisions (Arsal 2026-08-28, see `00_plan.md`):
- The invest team may write `InvestCustomer__pc`, `InvestmentStatus__pc`,
  `InvestmentExpirationDate__pc` and the invest consent, nothing else. Update rows
  carry only those.
- `consent_central` / `consent_camping` / `consent_residences` in the CSV are a
  mistake. Staged as 0, never read, never written, no reconciliation.
- `consent_invest=0` gets an OptOut consent, not a skip.
- Insert rows carry the full CSV row (new accounts are invest-owned).
- When one e-mail matches several accounts, the most recently modified account wins.
  The preview in section 5 needs sign-off (`ACCEPT_MULTI_MATCH_RULE`).

Prerequisites: fresh mirrors (`refresh_sf_mirrors.py`), `01_create_mirror_indexes.sql`
re-applied, recon numbers frozen via `02_recon_prereg.sql`.

In [13]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))  # repo root: config, mysql_client

from config import load_mysql_config
from mysql_client import MySQLClient

BATCH_DATE = "2026-08-28"
BATCHES = {
    ("update", 1): f"{BATCH_DATE}_prereg_update_optin",
    ("update", 0): f"{BATCH_DATE}_prereg_update_optout",
    ("insert", 1): f"{BATCH_DATE}_prereg_insert_optin",
    ("insert", 0): f"{BATCH_DATE}_prereg_insert_optout",
}
RAW_TABLE = "stg_imp_prereg_20260828"
CSV_PATH = Path.cwd() / "invest_preregistrations_2022-to-2026-08-25_exported-2026-08-26.csv"

db = MySQLClient(load_mysql_config())
print("connected | batches:", ", ".join(BATCHES.values()))

# Decision (Arsal 2026-08-28, multi-match review): office@daurer-reisen.at is a
# shared agency mailbox with 19 different people in the org; the newest-wins rule
# would update the wrong person. The row stages as _excluded=1 for manual clearing.
MANUAL_EXCLUDE = {
    "aae5f466-29cb-4cc5-b000-321af95cbef2": "shared_email_wrong_person",  # Patrick Daurer
}
EXCL_SQL = ",".join(f"'{e}'" for e in MANUAL_EXCLUDE) or "''"
EXCL_REASON_SQL = " ".join(
    f"WHEN s.external_id = '{e}' THEN '{r}'" for e, r in MANUAL_EXCLUDE.items()
)

connected | batches: 2026-08-28_prereg_update_optin, 2026-08-28_prereg_update_optout, 2026-08-28_prereg_insert_optin, 2026-08-28_prereg_insert_optout


## 1. Raw load: CSV into the stg table

Semicolon-delimited, UTF-8 BOM. Empty strings become NULL, and dates are parsed here
so the stage inserts stay plain SQL. The raw table is the recon anchor
(`02_recon_prereg.sql`). Re-running this cell drops and reloads it, which is safe as
long as nothing has been staged yet (section 3 guards that).

In [14]:
df = pd.read_csv(CSV_PATH, sep=";", encoding="utf-8-sig", dtype=str)
assert len(df) == 3193, f"expected 3,193 raw lines, got {len(df):,}"

df = df.apply(lambda s: s.str.strip()).replace({"": None})

# The export ends with a semicolons-only line that parses as an all-NaN row.
df = df.dropna(how="all")
assert len(df) == 3192, f"expected 3,192 data rows after dropping empty lines, got {len(df):,}"

for col in ("birth_date", "investment_expiration_date"):
    df[col] = pd.to_datetime(df[col], format="mixed", dayfirst=True, errors="raise").dt.date

for col in ("hotel_customer", "camping_customer", "residences_customer", "invest_customer",
            "consent_central", "consent_camping", "consent_residences", "consent_invest"):
    df[col] = df[col].fillna("0").astype(int)

assert df["invest_customer"].eq(1).all(), "non-invest row in an invest-team file"
assert df["external_id"].dropna().is_unique, "duplicate external_id"

db.execute(f"DROP TABLE IF EXISTS {RAW_TABLE}")
db.create_table_from_df(df, RAW_TABLE)
n = db.fetch_one(f"SELECT COUNT(*) AS n FROM {RAW_TABLE}")["n"]
print(f"{RAW_TABLE}: {n:,} rows | consent_invest=1: {df['consent_invest'].eq(1).sum():,} "
      f"| =0: {df['consent_invest'].eq(0).sum():,} | no email: {df['email'].isna().sum()}")

stg_imp_prereg_20260828: 3,192 rows | consent_invest=1: 3,192 | =0: 0 | no email: 0


## 2. Mirror freshness

`MAX(LastModifiedDate)` must be from today's refresh. A stale mirror mis-splits
update vs insert: a person created in the org yesterday would be staged as a new
account and duplicated.

In [15]:
row = db.fetch_one(
    "SELECT COUNT(*) AS n, MAX(LastModifiedDate) AS newest FROM crm_person_account_sfid_prod"
)
print(f"crm_person_account_sfid_prod  {row['n']:>12,}  newest: {row['newest']}")

crm_person_account_sfid_prod     1,497,812  newest: 2026-08-28 10:25:02


## 3. Batch guard + live schema check

Refuses to run if any of the four batch ids already has rows. Also verifies the
ALTER'd bookkeeping columns exist (the repo DDL is stale) and that
`uk_source_external_id (source, external_id)` has no leftovers from an earlier
batch that would make the stage INSERT collide.

In [16]:
for colname in ("_account_processed_at", "_consent_processed_at"):
    n = db.fetch_one("""
        SELECT COUNT(*) AS n FROM information_schema.columns
        WHERE table_schema = DATABASE()
          AND table_name = 'crm_imp_person_accounts' AND column_name = %s
    """, (colname,))["n"]
    assert n == 1, f"column {colname} missing on crm_imp_person_accounts"
print("bookkeeping columns present")

for b in BATCHES.values():
    existing = db.fetch_one(
        "SELECT COUNT(*) AS n FROM crm_imp_person_accounts WHERE _batch_id = %s", (b,)
    )["n"]
    assert existing == 0, f"{existing} rows already staged under {b}"
print("batch ids are free")

clash = db.fetch_one(f"""
    SELECT COUNT(*) AS n
    FROM crm_imp_person_accounts p
    JOIN {RAW_TABLE} s ON p.source = s.source AND p.external_id = s.external_id
""")["n"]
assert clash == 0, f"{clash} (source, external_id) collisions with rows already in staging"
print("no unique-key collisions")

bookkeeping columns present
batch ids are free
no unique-key collisions


## 4. Match map: raw row to existing account (or none)

`ExternalID__pc` matches first (recon says 2), then e-mail. E-mail matching compares
`LOWER(PersonEmail)`. When several accounts share the address, the most recently
modified one wins. The match lands in a temp table keyed by `external_id`.

In [17]:
# Real table, not TEMPORARY: MySQLClient opens a new connection per call,
# so a temp table would vanish before the next statement. Dropped in 03/09.
db.execute("DROP TABLE IF EXISTS tmp_prereg_match")
db.execute(f"""
    CREATE TABLE tmp_prereg_match AS
    SELECT s.external_id,
           COALESCE(x.Id, e.Id)                           AS sf_account_id,
           COALESCE(x.PersonContactId, e.PersonContactId) AS sf_person_contact_id,
           COALESCE(x.PersonEmail, e.PersonEmail)         AS org_email,
           CASE WHEN x.Id IS NOT NULL THEN 'external_id'
                WHEN e.Id IS NOT NULL THEN 'email' END    AS match_type,
           e.email_matches
    FROM {RAW_TABLE} s
    LEFT JOIN crm_person_account_sfid_prod x
           ON x.ExternalID__pc = s.external_id
    LEFT JOIN (
        SELECT s2.external_id AS ext_id, a.Id, a.PersonContactId, a.PersonEmail,
               COUNT(*) OVER (PARTITION BY s2.external_id)  AS email_matches,
               ROW_NUMBER() OVER (PARTITION BY s2.external_id
                                  ORDER BY a.LastModifiedDate DESC, a.Id) AS rn
        FROM {RAW_TABLE} s2
        JOIN crm_person_account_sfid_prod a
             ON LOWER(a.PersonEmail) = LOWER(s2.email)
        WHERE s2.email IS NOT NULL
    ) e ON e.ext_id = s.external_id AND e.rn = 1
""")
db.execute("CREATE INDEX idx_tmp_match_ext ON tmp_prereg_match (external_id)")
stats = db.fetch_one("""
    SELECT COUNT(*) AS total,
           SUM(match_type = 'external_id') AS by_extid,
           SUM(match_type = 'email')       AS by_email,
           SUM(match_type IS NULL)         AS unmatched,
           SUM(email_matches > 1)          AS multi_match
    FROM tmp_prereg_match
""")
for k, v in stats.items():
    print(f"{k:12s} {int(v or 0):,}")

total        3,192
by_extid     2
by_email     1,695
unmatched    1,495
multi_match  2


## 5. Multi-match preview, needs sign-off

Every e-mail that hits more than one account, with the candidate the rule picked
(newest `LastModifiedDate`). Review the export, then set
`ACCEPT_MULTI_MATCH_RULE = True`. The export contains PII and lands in the
gitignored `local_data/`.

In [18]:
ACCEPT_MULTI_MATCH_RULE = True   # flip after reviewing the export

multi = db.fetch_df(f"""
    SELECT s.external_id, s.email, s.first_name, s.last_name,
           a.Id AS candidate_account, a.FirstName AS org_first, a.LastName AS org_last,
           a.InvestCustomer__pc, a.SourceOrigin__pc, a.CreatedDate, a.LastModifiedDate,
           (a.Id = m.sf_account_id) AS picked
    FROM tmp_prereg_match m
    JOIN {RAW_TABLE} s ON s.external_id = m.external_id
    JOIN crm_person_account_sfid_prod a ON LOWER(a.PersonEmail) = LOWER(s.email)
    WHERE m.email_matches > 1
    ORDER BY s.email, a.LastModifiedDate DESC
""")
out = Path.cwd().parent / "local_data" / "prereg_multi_match_review.csv"
out.parent.mkdir(parents=True, exist_ok=True)
multi.to_csv(out, index=False)
print(f"{multi['external_id'].nunique():,} csv rows with >1 account match -> {out}")
assert ACCEPT_MULTI_MATCH_RULE, "review the multi-match export, then flip the flag" 

2 csv rows with >1 account match -> d:\repos\dev\Arsal\bi-crm-imports\local_data\prereg_multi_match_review.csv


## 6. Stage UPDATE rows (matched accounts)

One row per matched CSV row, `_operation='update'`, split by `consent_invest`.
Carries only what the update loader and the consent loader need: the three invest
columns, the sf ids, and audit fields. `sf_cp_email_id` joins in from the CPE mirror
via `PartyID__c = PersonContactId` plus the org e-mail, never a bare e-mail join.
Rows without a CPE stage with NULL and are excluded before the consent load
(section 8). Other-consent columns are forced to 0.

In [19]:
staged = {}
for flag in (1, 0):
    batch_id = BATCHES[("update", flag)]
    n = db.execute(f"""
        INSERT INTO crm_imp_person_accounts
            (_operation, _batch_id, _excluded, source, source_origin, external_id,
             email, first_name, last_name,
             sf_account_id, sf_person_contact_id, sf_cp_email_id,
             invest_customer, investment_status, investment_expiration_date,
             consent_invest, consent_central, consent_camping, consent_residences)
        SELECT
            'update', %s, 0, s.source, s.source_origin, s.external_id,
            s.email, s.first_name, s.last_name,
            m.sf_account_id, m.sf_person_contact_id, cpe.Id,
            s.invest_customer, s.investment_status, s.investment_expiration_date,
            s.consent_invest, 0, 0, 0
        FROM {RAW_TABLE} s
        JOIN tmp_prereg_match m ON m.external_id = s.external_id
        LEFT JOIN crm_cp_email_sfid_prod cpe
               ON  cpe.PartyID__c   = m.sf_person_contact_id
               AND cpe.EmailAddress = m.org_email
        WHERE m.sf_account_id IS NOT NULL
          AND s.external_id NOT IN ({EXCL_SQL})
          AND s.consent_invest = %s
    """, (batch_id, flag))
    staged[batch_id] = n
    print(f"staged {batch_id}: {n:,}")

staged 2026-08-28_prereg_update_optin: 1,696
staged 2026-08-28_prereg_update_optout: 0


## 7. Stage INSERT rows (net-new person accounts)

Full CSV row, `_operation='insert'`. Exclusions stage with `_excluded=1` and a
reason instead of being dropped, so the funnel stays auditable:
- no e-mail (0 expected: the only e-mail-less line in the export was the empty
  trailing line, dropped at raw load)
- non-ASCII local part (Salesforce rejects it with INVALID_EMAIL_ADDRESS)
- `MANUAL_EXCLUDE` rows from the multi-match review (the shared agency mailbox)

Non-ASCII domains stage normally. Salesforce punycodes them on insert, and the CPE
backfill in `04_run_prereg_accounts.ipynb` joins IDNA-encoded.

Country: the CSV carries ISO-2 codes and no street/city/postal. `PersonMailingCountry` gets the country NAME via `crm_tmp_country_names`, keeping invest accounts consistent with the nationality job's convention.

In [20]:
# Invest convention: PersonMailingCountry carries country NAMES (decision
# 2026-08-28, consistent with the nationality job). The CSV has ISO-2 codes;
# crm_tmp_country_names maps them. Guard: every non-NULL code must resolve.
unmapped = db.fetch_one(f"""
    SELECT COUNT(*) AS n FROM {RAW_TABLE} s
    LEFT JOIN crm_tmp_country_names cn ON cn.code = s.country
    WHERE s.country IS NOT NULL AND cn.name IS NULL
""")["n"]
assert unmapped == 0, f"{unmapped} country codes without a name mapping"

for flag in (1, 0):
    batch_id = BATCHES[("insert", flag)]
    n = db.execute(f"""
        INSERT INTO crm_imp_person_accounts
            (_operation, _batch_id, _excluded, _exclude_reason,
             source, source_origin, external_id, entra_external_id,
             salutation, first_name, middle_name, last_name,
             birth_date, birth_place, gender, email, phone,
             preferred_language, nationality_country_code,
             address, postal_code, city, state, country,
             hotel_customer, camping_customer, residences_customer, invest_customer,
             investment_status, investment_expiration_date,
             consent_invest, consent_central, consent_camping, consent_residences)
        SELECT
            'insert', %s,
            CASE WHEN s.external_id IN ({EXCL_SQL}) THEN 1
                 WHEN s.email IS NULL THEN 1
                 WHEN SUBSTRING_INDEX(s.email, '@', 1)
                      <> CONVERT(SUBSTRING_INDEX(s.email, '@', 1) USING ascii) THEN 1
                 ELSE 0 END,
            CASE {EXCL_REASON_SQL}
                 WHEN s.email IS NULL THEN 'no_email'
                 WHEN SUBSTRING_INDEX(s.email, '@', 1)
                      <> CONVERT(SUBSTRING_INDEX(s.email, '@', 1) USING ascii) THEN 'non_ascii_local_part'
                 ELSE NULL END,
            s.source, s.source_origin, s.external_id, s.entra_external_id,
            s.salutation, s.first_name, s.middle_name, s.last_name,
            s.birth_date, s.birth_place, s.gender, s.email, s.phone,
            s.preferred_language, s.nationality_country_code,
            s.address, s.postal_code, s.city, s.state, cn.name,
            0, 0, 0, s.invest_customer,
            s.investment_status, s.investment_expiration_date,
            s.consent_invest, 0, 0, 0
        FROM {RAW_TABLE} s
        LEFT JOIN crm_tmp_country_names cn ON cn.code = s.country
        LEFT JOIN tmp_prereg_match m ON m.external_id = s.external_id
        WHERE (m.sf_account_id IS NULL OR s.external_id IN ({EXCL_SQL}))
          AND s.consent_invest = %s
    """, (batch_id, flag))
    staged[batch_id] = n
    print(f"staged {batch_id}: {n:,}")

staged 2026-08-28_prereg_insert_optin: 1,496
staged 2026-08-28_prereg_insert_optout: 0


## 8. Final contract

Every CSV row lands in exactly one batch, update rows all carry `sf_account_id`, no
account is staged twice, and the other-consent columns are 0 everywhere. Copy the
per-batch counts into `04_run_prereg_accounts.ipynb` (`EXPECTED`). They are the load
contract.

In [21]:
ids = tuple(BATCHES.values())
summary = db.fetch_one(f"""
    SELECT COUNT(*) AS rows_staged,
           SUM(_operation = 'update') AS upd,
           SUM(_operation = 'insert') AS ins,
           SUM(_excluded = 1) AS excluded,
           SUM(_operation = 'update' AND sf_account_id IS NULL) AS upd_without_id,
           SUM(_operation = 'update' AND sf_cp_email_id IS NULL) AS upd_without_cpe,
           SUM(consent_central + consent_camping + consent_residences) AS other_consents,
           COUNT(DISTINCT external_id) AS uniq_ext
    FROM crm_imp_person_accounts
    WHERE _batch_id IN {ids!r}
""")
for k, v in summary.items():
    print(f"{k:18s} {int(v or 0):,}")

total_raw = db.fetch_one(f"SELECT COUNT(*) AS n FROM {RAW_TABLE}")["n"]
assert int(summary["rows_staged"]) == total_raw, "row lost between raw and staging"
assert int(summary["upd_without_id"]) == 0, "update row without sf_account_id"
assert int(summary["other_consents"]) == 0, "a non-invest consent flag leaked through"

dup = db.fetch_one(f"""
    SELECT COUNT(*) AS n FROM (
        SELECT sf_account_id FROM crm_imp_person_accounts
        WHERE _batch_id IN {ids!r} AND sf_account_id IS NOT NULL
        GROUP BY sf_account_id HAVING COUNT(*) > 1
    ) d
""")["n"]
assert dup == 0, f"{dup} accounts staged more than once"

per_batch = db.fetch_all(f"""
    SELECT _batch_id, COUNT(*) AS n, SUM(_excluded = 1) AS excl
    FROM crm_imp_person_accounts WHERE _batch_id IN {ids!r}
    GROUP BY _batch_id ORDER BY _batch_id
""")
print()
for r in per_batch:
    print(f"  {r['_batch_id']}: {int(r['n']):,} rows ({int(r['excl'])} excluded)")

no_cpe = int(summary["upd_without_cpe"])
print(f"\nupdate rows without CPE (consent needs the post-refresh backfill or exclusion): {no_cpe:,}")
print("staging frozen")

rows_staged        3,192
upd                1,696
ins                1,496
excluded           1
upd_without_id     0
upd_without_cpe    0
other_consents     0
uniq_ext           3,192

  2026-08-28_prereg_insert_optin: 1,496 rows (1 excluded)
  2026-08-28_prereg_update_optin: 1,696 rows (0 excluded)

update rows without CPE (consent needs the post-refresh backfill or exclusion): 0
staging frozen


## 9. Review exports (no writes)

Both exports contain PII and land in the gitignored `local_data/`: the excluded
rows, and the update rows whose account has no CPE yet (their consent depends on
the backfill in 04 section 6).

In [22]:
ids = tuple(BATCHES.values())
excluded = db.fetch_df(f"""
    SELECT _batch_id, _exclude_reason, external_id, email, first_name, last_name
    FROM crm_imp_person_accounts
    WHERE _batch_id IN {ids!r} AND _excluded = 1
""")
out = Path.cwd().parent / "local_data" / "prereg_excluded_review.csv"
excluded.to_csv(out, index=False)
print(f"{len(excluded):,} excluded rows -> {out}")

no_cpe = db.fetch_df(f"""
    SELECT _batch_id, external_id, email, sf_account_id, sf_person_contact_id
    FROM crm_imp_person_accounts
    WHERE _batch_id IN {ids!r} AND _operation = 'update' AND sf_cp_email_id IS NULL
""")
out = Path.cwd().parent / "local_data" / "prereg_update_no_cpe_review.csv"
no_cpe.to_csv(out, index=False)
print(f"{len(no_cpe):,} update rows without CPE -> {out}")

db.execute("DROP TABLE IF EXISTS tmp_prereg_match")
print("match table dropped (staging is frozen; 04/05 work from crm_imp_person_accounts)")

1 excluded rows -> d:\repos\dev\Arsal\bi-crm-imports\local_data\prereg_excluded_review.csv
0 update rows without CPE -> d:\repos\dev\Arsal\bi-crm-imports\local_data\prereg_update_no_cpe_review.csv
match table dropped (staging is frozen; 04/05 work from crm_imp_person_accounts)


## Next

- `04_run_prereg_accounts.ipynb`: dry-runs, probes, account update and insert loads,
  id writebacks, mirror refresh, CPE backfill.
- `05_run_prereg_consents.ipynb`: consent dry-runs, probe, load, verify, archive.
- Loads do not run without Arsal's explicit go-ahead (`RUN_PROBE` / `RUN_LOAD` gates).